# Experimentos

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import pandas as pd
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from data import get_training_data

mlflow.set_tracking_uri("file:" + os.path.join(os.getcwd(), "mlruns"))
mlflow.set_experiment("stock-direction")

## Datos y partición

In [ ]:
X, y, feature_names = get_training_data()

# split temporal: 80% entrena, 20% prueba, sin shuffle
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

majority = y_train.mode()[0]
baseline_acc = (y_test == majority).mean()
print("Entrenamiento:", len(X_train), "| Prueba:", len(X_test))
print("Baseline:", round(baseline_acc, 4))

## Modelos

In [ ]:
def entrenar_y_registrar(modelo, nombre, params=None):
    with mlflow.start_run(run_name=nombre):
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", modelo)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        proba = pipe.predict_proba(X_test)[:, 1]
        metrics = {
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, proba),
        }
        mlflow.log_param("modelo", nombre)
        if params:
            mlflow.log_params(params)
        mlflow.log_metric("baseline_accuracy", baseline_acc)
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
        return metrics

In [ ]:
m1 = entrenar_y_registrar(
    LogisticRegression(max_iter=1000, C=1.0),
    "logistic_regression",
    {"C": 1.0},
)
m1

In [ ]:
m2 = entrenar_y_registrar(
    RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "random_forest",
    {"n_estimators": 200, "max_depth": 5},
)
m2

In [ ]:
m3 = entrenar_y_registrar(
    GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
    "gradient_boosting",
    {"n_estimators": 200, "max_depth": 3},
)
m3

## Comparación

In [ ]:
pd.DataFrame(
    [m1, m2, m3],
    index=["logistic_regression", "random_forest", "gradient_boosting"],
)

## Modelo elegido
